In [1]:
!pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb -q

In [2]:
from langchain_core.documents import Document

Document -> Object

{

    1. text/page content
    
    2. metadata
    
}

In [3]:
# Text data

from langchain_community.document_loaders.text import TextLoader

loader = TextLoader("/kaggle/input/datasets/radhikaasmar/data-rag/Python.txt", encoding="utf-8")

document = loader.load()
document

[Document(metadata={'source': '/kaggle/input/datasets/radhikaasmar/data-rag/Python.txt'}, page_content='\ufeffPython is a high-level, interpreted programming language that has become one of the most popular and widely used languages in the world. Created by Guido van Rossum and first released in 1991, Python emphasizes simplicity and readability, making it easy for beginners to learn while remaining powerful for experienced developers. Its clean and concise syntax allows programmers to write fewer lines of code compared to many other languages, enhancing productivity and maintainability. Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming, which makes it versatile for a wide range of applications.\nSome key features and benefits of Python include:\n* Ease of Learning: Simple syntax and readability make Python beginner-friendly.\n* Versatility: Suitable for web development, data analysis, artificial intelligence, machine lear

In [4]:
# # PDF data

# from langchain_community.document_loaders.pdf import PyPDFLoader

# pdf_loader = PyPDFLoader("/kaggle/input/datasets/radhikaasmar/data-rag/research.pdf")

# document = pdf_loader.load()
# document

In [5]:
# # PDF data

# from langchain_community.document_loaders.pdf import PyMuPDFLoader

# pdf_loader = PyMuPDFLoader("/kaggle/input/datasets/radhikaasmar/data-rag/research.pdf")

# document = pdf_loader.load()
# document

# 1. Injestion Pipeline

In [6]:
# Data => Documents

import os
from langchain_community.document_loaders.pdf import PyPDFLoader

## i. Documents

In [7]:
def load_all_pdfs():
    folder_path = "/kaggle/input/datasets/radhikaasmar/data-rag/pdfs/pdfs"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            pdf_path = os.path.join(folder_path, filename)

            loader = PyPDFLoader(pdf_path)
            doc = loader.load()

            all_docs.extend(doc)
            num_docs += 1

    print("total pdfs:", num_docs)
    print("total pages:", len(all_docs))
    return all_docs

In [8]:
all_pdf_documents = load_all_pdfs()

total pdfs: 2
total pages: 32


In [9]:
type(all_pdf_documents[0])

langchain_core.documents.base.Document

In [10]:
all_pdf_documents[0]

Document(metadata={'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'publisher': 'Curran Associates, Inc.', 'language': 'en-US', 'created': '2017', 'eventtype': 'Poster', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly less timeto train. Our single model with 165 million parameters, achieves 27.5 BLEU onEnglish-to-German translation, improving over the existing best ensemble result by over 1 BLEU. On E

## ii. Chunks

In [11]:
# Chunks 

!pip install langchain_text_splitters -q

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents, chunk_size=500, chunk_overlap=50):

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )

    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs

In [13]:
chunks = split_docs(all_pdf_documents)
len(chunks)

320

In [14]:
chunks[0]

Document(metadata={'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'publisher': 'Curran Associates, Inc.', 'language': 'en-US', 'created': '2017', 'eventtype': 'Poster', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly less timeto train. Our single model with 165 million parameters, achieves 27.5 BLEU onEnglish-to-German translation, improving over the existing best ensemble result by over 1 BLEU. On E

## iii. Embedding

In [15]:
from sentence_transformers import SentenceTransformer

In [16]:
from sentence_transformers import SentenceTransformer

class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model_name = model_name
        print("loading model....", self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("embedding dimensions=", self.model.get_sentence_embedding_dimension())

    def generate_embeddings(self, text):
        # Generates numerical representations (embeddings) for the input text
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("embeddings shape:", embeddings.shape)
        return embeddings

In [17]:
# Initialize the manager
embedding_manager = EmbeddingManager()

loading model.... all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


embedding dimensions= 384


## iv. Vector Store 

Small local version of Vector DB

* initialize vector store - constructor
* vs_initialize -> path, collection
* store docs -> Vector Store

In [18]:
!pip install opentelemetry-api==1.24.0 opentelemetry-sdk==1.24.0 -q

In [19]:
import chromadb
import uuid
import opentelemetry.context
import uuid

In [20]:
class VectorStoreManager:
    def __init__(self, persist_directory="data/vector_store", collection_name="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None

        self._initialize_store()

    def _initialize_store(self):
        # Ensure the directory where data will be saved actually exists
        os.makedirs(self.persist_directory, exist_ok=True)

        # Create a client that saves data to the local hard drive
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        # Create the collection (or open it if it already exists)
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "vector store collection for pdf embeddings in RAG"}
        )

        print("initialized the vector store with collection:", self.collection_name)
        print("docs in collection:", self.collection.count())


    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("num of documents does not match num of embeddings")
        
        # store => ids, embedding, document, metadata
        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate a unique ID for every single chunk
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)
            
            # Prepare metadata (useful for filtering later)
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)
            
            # Extract the text and convert embedding to list format
            documents_content.append(doc.page_content)
            embeddings_list.append(embedding.tolist())
            
        # Final step: Upload everything to the ChromaDB collection
        self.collection.add(
            ids=ids,
            metadatas=all_metadata,
            documents=documents_content,
            embeddings=embeddings_list
        )
        
        print("total documents added in vector store=", len(documents_content))
        print("docs in collection:", self.collection.count())


In [21]:
vector_store = VectorStoreManager()

initialized the vector store with collection: pdf_documents
docs in collection: 320


In [22]:
# data => documents => chunks => embeddings => store in vector store

# 1. Extract the raw text from your document chunks
texts = [doc.page_content for doc in chunks]

# 2. Use your EmbeddingManager to turn those texts into numerical vectors
embedding = embedding_manager.generate_embeddings(texts)

# 3. Save both the original chunks and their new embeddings into the database
vector_store.add_documents(chunks, embedding)

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

embeddings shape: (320, 384)
total documents added in vector store= 320
docs in collection: 640


# 2. Retrieval Pipeline

In [23]:
from sklearn.metrics.pairwise import cosine_similarity

In [24]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        # Store the managers we built earlier so we can use their methods
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    def retrieve(self, query, top_k=5, score_threshold=0.0):
        """
        Searches the vector database for the most relevant document chunks.
        
        Args:
            query (str): The user's natural language question.
            top_k (int): Number of documents to return.
            score_threshold (float): Minimum similarity (0 to 1) to be considered a match.
        """
        
        # 1. Convert the search query into a vector (embedding)
        # We pass [query] as a list and take the first result [0]
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        # 2. Perform the semantic search in ChromaDB
        # We convert the numpy array to a list so ChromaDB can read it
        results = self.vector_store.collection.query(
            query_embeddings=[query_embeddings.tolist()],
            n_results=top_k
        )

        retrieved_docs = []
        
        # 3. Check if we actually found anything
        if results["documents"] and results["documents"][0]:
            # ChromaDB returns nested lists; we grab the inner lists
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            # 4. Loop through results and calculate a similarity score
            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                
                # Distance measures how 'far apart' vectors are. 
                # Similarity (1 - distance) measures how 'close' they are.
                similarity_score = 1 - distance

                # Only keep the document if it meets our quality bar (threshold)
                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "document": document,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank": i + 1  # 1 is the best match, 2 is the second best, etc.
                    })

            print(f"retrieved {len(retrieved_docs)} documents")

        else:
            print("no documents found")

        return retrieved_docs

In [25]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [26]:
rag_retriever.retrieve("What is RAG?")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 5 documents


[{'id': 'doc_770e02e4-1ec3-4a40-8ea1-861bbea511d0',
  'document': 'and speculate on upcoming trends and innovations.\nOur contributions are as follows:\n• In this survey, we present a thorough and systematic\nreview of the state-of-the-art RAG methods, delineating\nits evolution through paradigms including naive RAG,\narXiv:2312.10997v5  [cs.CL]  27 Mar 2024',
  'metadata': {'trapped': '/False',
   'page_label': '1',
   'author': '',
   'creator': 'LaTeX with hyperref',
   'moddate': '2024-03-28T00:54:45+00:00',
   'title': '',
   'keywords': '',
   'content_length': 288,
   'subject': '',
   'creationdate': '2024-03-28T00:54:45+00:00',
   'total_pages': 21,
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
   'page': 0,
   'producer': 'pdfTeX-1.40.25',
   'doc_index': 87,
   'source': '/kaggle/input/datasets/radhikaasmar/data-rag/pdfs/pdfs/research2.pdf'},
  'distance': 0.46291497349739075,
  'similarity_score': 0.537085026

In [27]:
rag_retriever.retrieve("What is encoder decoder")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 5 documents


[{'id': 'doc_d4d31d9c-6944-4c67-b00c-59077e6b1c1f',
  'document': 'positional encodings in both the encoder and decoder stacks. For the base model, we use a rate of\nPdrop = 0.1.\n7',
  'metadata': {'language': 'en-US',
   'date': '2017',
   'publisher': 'Curran Associates, Inc.',
   'page_label': '7',
   'content_length': 112,
   'creationdate': '',
   'title': 'Attention is All you Need',
   'book': 'Advances in Neural Information Processing Systems 30',
   'total_pages': 11,
   'creator': 'PyPDF',
   'type': 'Conference Proceedings',
   'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation t